<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-04-image-and-multimodal-generation/lesson-4.7-voice-ai-stt-and-tts/notebooks/exercises-4_7.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 4.7: Voice AI: Speech-to-Text & Text-to-Speech

*Module 4 · Image, Vision & Multimodal AI*

> Speech-to-Text turns spoken words into data your AI can process. Text-to-Speech gives your AI a natural human voice. Together they enable voice assistants, call center bots, podcast transcription, and multilingual dubbing.

---

## About this notebook

This notebook contains the **7 runnable code examples** from the published lesson `lesson-4.7.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Speech-to-Text — Turning Voice into Data — `part1a_whisper_local.py`
2. Step 1 — Speech-to-Text — Turning Voice into Data — `part1b_google_stt.py`
3. Step 2 — Chirp — Google's Universal Speech Model for India — `part2_chirp_indian_languages.py`
4. Step 3 — Text-to-Speech — Give Your AI a Natural Voice — `part3a_google_tts.py`
5. Step 3 — Text-to-Speech — Give Your AI a Natural Voice
6. Step 4 — Real-Time Streaming — Transcribe as You Speak — `part4_streaming_stt.py`
7. Step 5 — Project: Voice-Enabled AI Assistant — `project_voice_assistant.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai openai torch


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Speech-to-Text — Turning Voice into Data

Record audio, send it to an API, get text back. The foundation of all voice AI.

**`part1a_whisper_local.py`** · _Block 1 of 7_

WHISPER: OpenAI's open-source STT model — Runs locally. Free. Supports 99 languages.


In [ ]:
# =============================================
# WHISPER: OpenAI's open-source STT model
# Runs locally. Free. Supports 99 languages.
# pip install openai-whisper torch
# =============================================

import whisper
import time

# Load model (sizes: tiny, base, small, medium, large-v3)
print("Loading Whisper model...")
model = whisper.load_model("base")  # 74M params, good balance of speed/accuracy

# Transcribe an audio file
start = time.time()
result = model.transcribe("meeting_recording.mp3")
elapsed = time.time() - start

print(f"Transcription ({elapsed:.1f}s):\n")
print(result["text"])

# ── With timestamps (for subtitles) ──
print("\nWith timestamps:")
for segment in result["segments"]:
    start_t = segment["start"]
    end_t = segment["end"]
    text = segment["text"]
    print(f"  [{start_t:6.1f}s - {end_t:6.1f}s] {text}")

# ── Detect language automatically ──
print(f"\nDetected language: {result['language']}")

print("""
Whisper model sizes:
  tiny   (39M)  — fastest, least accurate. Good for quick tests.
  base   (74M)  — good speed/accuracy balance. Recommended to start.
  small  (244M) — noticeably better for accented speech.
  medium (769M) — great for Indian English and Hindi.
  large-v3 (1.5B) — best accuracy. Needs GPU.
  
Whisper handles: English, Hindi, Tamil, Telugu, Bengali, Marathi,
  Gujarati, Kannada, Malayalam, Punjabi, and 90+ more languages.
""")


**`part1b_google_stt.py`** · _Block 2 of 7_

GOOGLE CLOUD SPEECH-TO-TEXT V2 — Production-grade, supports Chirp 2 model,


In [ ]:
# =============================================
# GOOGLE CLOUD SPEECH-TO-TEXT V2
# Production-grade, supports Chirp 2 model,
# speaker diarization, word-level timestamps.
# pip install google-cloud-speech
# =============================================

from google.cloud import speech_v2 as speech
from google.cloud.speech_v2.types import cloud_speech

# Create client
client = speech.SpeechClient()

# Read audio file
with open("customer_call.wav", "rb") as f:
    audio_data = f.read()

# Configure recognition
config = cloud_speech.RecognitionConfig(
    auto_decoding_config=cloud_speech.AutoDetectDecodingConfig(),
    language_codes=["en-IN", "hi-IN"],  # English-India + Hindi (handles code-switching!)
    model="chirp_2",                    # Google's latest universal speech model
    features=cloud_speech.RecognitionFeatures(
        enable_word_time_offsets=True,   # timestamp for every word
        enable_automatic_punctuation=True,
        enable_spoken_punctuation=True,
    ),
)

request = cloud_speech.RecognizeRequest(
    recognizer=f"projects/YOUR_PROJECT/locations/global/recognizers/_",
    config=config,
    content=audio_data,
)

# Transcribe
response = client.recognize(request=request)

print("Google Cloud STT V2 + Chirp 2:\n")
for result in response.results:
    alt = result.alternatives[0]
    print(f"  Text: {alt.transcript}")
    print(f"  Confidence: {alt.confidence:.1%}")
    
    # Word-level timestamps
    for word in alt.words[:5]:  # show first 5 words
        start = word.start_offset.total_seconds()
        end = word.end_offset.total_seconds()
        print(f"    [{start:.2f}s] {word.word}")


> **What just happened?** Two STT approaches: (1) Whisper — open-source, runs locally, free, supports 99 languages. The "base" model handles Indian English and Hindi well; "large-v3" is best for accuracy. (2) Google Cloud STT V2 with Chirp 2 — production-grade, handles Hindi-English code-switching (mixing languages mid-sentence), provides word-level timestamps and automatic punctuation. Use Whisper for prototyping and privacy; Google Cloud for production quality.


### Step 2 · Chirp — Google's Universal Speech Model for India

Handles Hindi, Tamil, Telugu, Bengali, Marathi, and code-switching — the real Indian voice challenge.

**`part2_chirp_indian_languages.py`** · _Block 3 of 7_

CHIRP 2: Multilingual Indian speech recognition — Handles code-switching (Hindi + English mixed)


In [ ]:
# =============================================
# CHIRP 2: Multilingual Indian speech recognition
# Handles code-switching (Hindi + English mixed)
# =============================================

from google.cloud import speech_v2 as speech
from google.cloud.speech_v2.types import cloud_speech

client = speech.SpeechClient()

def transcribe_indian(audio_path: str, languages: list[str]) -> dict:
    """Transcribe audio with Indian language support."""
    
    with open(audio_path, "rb") as f:
        audio_data = f.read()
    
    config = cloud_speech.RecognitionConfig(
        auto_decoding_config=cloud_speech.AutoDetectDecodingConfig(),
        language_codes=languages,
        model="chirp_2",
        features=cloud_speech.RecognitionFeatures(
            enable_automatic_punctuation=True,
            enable_word_time_offsets=True,
        ),
    )
    
    request = cloud_speech.RecognizeRequest(
        recognizer=f"projects/YOUR_PROJECT/locations/global/recognizers/_",
        config=config,
        content=audio_data,
    )
    
    response = client.recognize(request=request)
    
    results = []
    for r in response.results:
        alt = r.alternatives[0]
        results.append({
            "text": alt.transcript,
            "confidence": alt.confidence,
            "language": r.language_code,
        })
    
    return {
        "full_text": " ".join(r["text"] for r in results),
        "segments": results,
    }

# ── Test with different Indian language scenarios ──

# Hindi-English code-switching (most common in urban India)
print("Test 1: Hindi-English code-switching")
result = transcribe_indian("hinglish_call.wav", ["hi-IN", "en-IN"])
print(f"  → {result['full_text']}\n")

# Pure Hindi
print("Test 2: Pure Hindi")
result = transcribe_indian("hindi_audio.wav", ["hi-IN"])
print(f"  → {result['full_text']}\n")

# Telugu
print("Test 3: Telugu")
result = transcribe_indian("telugu_audio.wav", ["te-IN"])
print(f"  → {result['full_text']}\n")

# Tamil
print("Test 4: Tamil")
result = transcribe_indian("tamil_audio.wav", ["ta-IN"])
print(f"  → {result['full_text']}\n")

print("""
Supported Indian languages in Chirp 2:
  hi-IN  Hindi          bn-IN  Bengali       gu-IN  Gujarati
  ta-IN  Tamil          te-IN  Telugu        kn-IN  Kannada
  ml-IN  Malayalam      mr-IN  Marathi       pa-IN  Punjabi
  or-IN  Odia           as-IN  Assamese      ur-IN  Urdu
  
The killer feature: multi-language codes ["hi-IN", "en-IN"]
allows code-switching — the model auto-detects which language 
is being spoken at each moment, even mid-sentence.
""")


> **What just happened?** We tested Chirp 2 on 4 Indian language scenarios: Hindi-English code-switching, pure Hindi, Telugu, and Tamil. The code-switching test is the critical one — Indian users naturally mix Hindi and English, and Chirp handles this seamlessly by accepting multiple language codes. It auto-detects which language is spoken at each moment. This is essential for any voice AI deployed in India.


### Step 3 · Text-to-Speech — Give Your AI a Natural Voice

Convert text to spoken audio. Choose from dozens of voices, languages, and styles.

**`part3a_google_tts.py`** · _Block 4 of 7_

GOOGLE CLOUD TEXT-TO-SPEECH — Natural voices in 50+ languages, including


In [ ]:
# =============================================
# GOOGLE CLOUD TEXT-TO-SPEECH
# Natural voices in 50+ languages, including
# multiple Indian language voices.
# pip install google-cloud-texttospeech
# =============================================

from google.cloud import texttospeech

client = texttospeech.TextToSpeechClient()

def speak(text: str, output_file: str,
         language: str = "en-IN",
         voice_name: str = "en-IN-Wavenet-D",  # Indian English male
         speed: float = 1.0,
         pitch: float = 0.0):
    """Convert text to natural speech and save as MP3."""
    
    synthesis_input = texttospeech.SynthesisInput(text=text)
    
    voice = texttospeech.VoiceSelectionParams(
        language_code=language,
        name=voice_name,
    )
    
    audio_config = texttospeech.AudioConfig(
        audio_encoding=texttospeech.AudioEncoding.MP3,
        speaking_rate=speed,     # 0.5 = slow, 1.0 = normal, 2.0 = fast
        pitch=pitch,             # -10 = deeper, 0 = normal, +10 = higher
        effects_profile_id=["headphone-class-device"],  # audio optimization
    )
    
    response = client.synthesize_speech(
        input=synthesis_input, voice=voice, audio_config=audio_config
    )
    
    with open(output_file, "wb") as f:
        f.write(response.audio_content)
    
    print(f"  Saved: {output_file}")

# ── Generate speech in multiple Indian voices ──
text = "Welcome to Netsetos. Your AI learning journey starts here. Let's build something amazing together."

voices = [
    ("en-IN-Wavenet-D", "en-IN", "Indian English Male"),
    ("en-IN-Wavenet-A", "en-IN", "Indian English Female"),
    ("hi-IN-Wavenet-A", "hi-IN", "Hindi Female"),
    ("hi-IN-Wavenet-B", "hi-IN", "Hindi Male"),
    ("te-IN-Standard-A", "te-IN", "Telugu Female"),
    ("ta-IN-Wavenet-A", "ta-IN", "Tamil Female"),
]

hindi_text = "नेटसेटोस में आपका स्वागत है। आपकी AI सीखने की यात्रा यहीं से शुरू होती है।"

print("Generating speech samples:\n")
for voice_name, lang, label in voices:
    print(f"  {label}:")
    content = hindi_text if "hi-IN" in lang else text
    speak(content, f"voice_{label.replace(' ', '_').lower()}.mp3", lang, voice_name)

print("""
Voice tiers (quality & pricing):
  Standard  — decent quality, cheapest ($4/1M characters)
  Wavenet   — very natural, uses DeepMind's WaveNet ($16/1M chars)
  Neural2   — latest, most natural ($16/1M chars)
  Studio    — professional quality, limited languages ($160/1M chars)
  
Recommendation: Start with Wavenet for Indian English/Hindi.
  Use Neural2 for premium quality where available.
""")


_Block 5 of 7_

SSML: Control HOW the AI speaks — Add pauses, emphasis, speed changes, and more.


In [ ]:
# =============================================
# SSML: Control HOW the AI speaks
# Add pauses, emphasis, speed changes, and more.
# =============================================

def speak_ssml(ssml: str, output_file: str, voice_name="en-IN-Wavenet-D", lang="en-IN"):
    """Generate speech from SSML (Speech Synthesis Markup Language)."""
    synthesis_input = texttospeech.SynthesisInput(ssml=ssml)
    voice = texttospeech.VoiceSelectionParams(language_code=lang, name=voice_name)
    audio_config = texttospeech.AudioConfig(audio_encoding=texttospeech.AudioEncoding.MP3)
    
    response = client.synthesize_speech(input=synthesis_input, voice=voice, audio_config=audio_config)
    with open(output_file, "wb") as f:
        f.write(response.audio_content)
    print(f"  Saved: {output_file}")

# SSML lets you control delivery with XML tags
ssml = """<speak>
  Welcome to <emphasis level="strong">Netsetos</emphasis>.
  
  <break time="500ms"/>
  
  Today we'll cover <prosody rate="slow">three important topics</prosody>.
  
  <break time="300ms"/>
  
  First: <prosody pitch="+2st">Speech to Text.</prosody>
  Second: <prosody pitch="+1st">Text to Speech.</prosody>
  Third: <prosody pitch="+3st" rate="fast">Building a complete voice assistant!</prosody>
  
  <break time="1s"/>
  
  Let's get started.
</speak>"""

speak_ssml(ssml, "ssml_demo.mp3")

print("""
SSML tags you'll use most:
  <break time="500ms"/>     — pause for half a second
  <emphasis level="strong"> — stress a word
  <prosody rate="slow">     — speak slowly (or "fast", "x-fast")
  <prosody pitch="+2st">    — raise pitch by 2 semitones
  <say-as interpret-as="cardinal">42</say-as> — say "forty-two"
  <say-as interpret-as="date">2026-04-03</say-as> — say "April third"
""")


> **What just happened?** We generated speech in 6 different voices (Indian English male/female, Hindi male/female, Telugu, Tamil), then learned SSML — a markup language that controls how the AI speaks: adding pauses, emphasis, speed changes, and pitch shifts. SSML is what makes the difference between a robotic reading and a natural-sounding presentation. Voice tiers range from Standard ($4/1M chars) to Studio ($160/1M chars).


### Step 4 · Real-Time Streaming — Transcribe as You Speak

Don't wait for the recording to finish. Get text AS the person talks — word by word.

**`part4_streaming_stt.py`** · _Block 6 of 7_

REAL-TIME STT with Gemini Live API — Transcribe microphone audio as you speak.


In [ ]:
# =============================================
# REAL-TIME STT with Gemini Live API
# Transcribe microphone audio as you speak.
# pip install google-generativeai pyaudio
# =============================================

import google.generativeai as genai
import pyaudio
import wave
import io

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

# ── Record from microphone ──
def record_audio(duration=5, sample_rate=16000):
    """Record audio from microphone for specified duration."""
    audio = pyaudio.PyAudio()
    stream = audio.open(
        format=pyaudio.paInt16,
        channels=1,
        rate=sample_rate,
        input=True,
        frames_per_buffer=1024,
    )
    
    print(f"Recording for {duration} seconds... Speak now!")
    frames = []
    for _ in range(int(sample_rate / 1024 * duration)):
        data = stream.read(1024)
        frames.append(data)
    
    stream.stop_stream()
    stream.close()
    audio.terminate()
    print("Recording complete!")
    
    # Convert to WAV bytes
    buffer = io.BytesIO()
    with wave.open(buffer, "wb") as wf:
        wf.setnchannels(1)
        wf.setsampwidth(audio.get_sample_size(pyaudio.paInt16))
        wf.setframerate(sample_rate)
        wf.writeframes(b"".join(frames))
    
    return buffer.getvalue()

# ── Transcribe with Gemini ──
model = genai.GenerativeModel("gemini-2.0-flash")

print("Voice-to-text demo:\n")
audio_bytes = record_audio(duration=5)

# Send audio directly to Gemini for transcription + understanding
response = model.generate_content([
    "Transcribe this audio. Then answer any question asked in it.",
    {"mime_type": "audio/wav", "data": audio_bytes},
])

print(f"Gemini heard: {response.text}")


> **What just happened?** We recorded 5 seconds of audio from the microphone and sent it to Gemini, which both transcribed it AND understood its meaning. Unlike traditional STT that only converts speech to text, Gemini processes the audio directly and can answer questions, summarize, translate, or analyze the tone — all from raw audio. This is the "native multimodal" advantage: audio goes straight into the model, no STT pre-processing step needed.


### Step 5 · Project: Voice-Enabled AI Assistant

Speak a question → AI thinks → you hear the answer. Complete loop.

**`project_voice_assistant.py`** · _Block 7 of 7_

VOICE AI ASSISTANT: Speak → Think → Respond — Complete voice loop using Gemini + Google TTS.


In [ ]:
# =============================================
# VOICE AI ASSISTANT: Speak → Think → Respond
# Complete voice loop using Gemini + Google TTS.
# =============================================

import google.generativeai as genai
from google.cloud import texttospeech
import pyaudio, wave, io, os, subprocess

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class VoiceAssistant:
    """A voice AI assistant: listen → think → speak."""
    
    def __init__(self, persona="a helpful assistant that gives concise answers in 2-3 sentences"):
        self.gemini = genai.GenerativeModel(
            "gemini-2.0-flash",
            system_instruction=f"You are {persona}. Keep answers short and conversational — they will be read aloud."
        )
        self.tts_client = texttospeech.TextToSpeechClient()
        self.chat = self.gemini.start_chat()
    
    def listen(self, duration=5) -> bytes:
        """Record audio from microphone."""
        audio = pyaudio.PyAudio()
        stream = audio.open(format=pyaudio.paInt16, channels=1,
                           rate=16000, input=True, frames_per_buffer=1024)
        
        print("  🎤 Listening...")
        frames = []
        for _ in range(int(16000 / 1024 * duration)):
            frames.append(stream.read(1024))
        
        stream.stop_stream()
        stream.close()
        audio.terminate()
        
        buffer = io.BytesIO()
        with wave.open(buffer, "wb") as wf:
            wf.setnchannels(1)
            wf.setsampwidth(2)
            wf.setframerate(16000)
            wf.writeframes(b"".join(frames))
        return buffer.getvalue()
    
    def think(self, audio_bytes: bytes) -> str:
        """Send audio to Gemini → get text response."""
        print("  🧠 Thinking...")
        response = self.chat.send_message([
            "Listen to this audio and respond to what the person said.",
            {"mime_type": "audio/wav", "data": audio_bytes},
        ])
        return response.text
    
    def speak(self, text: str):
        """Convert text to speech and play it."""
        print(f"  🔊 Speaking: {text[:80]}...")
        
        synthesis_input = texttospeech.SynthesisInput(text=text)
        voice = texttospeech.VoiceSelectionParams(
            language_code="en-IN",
            name="en-IN-Wavenet-D",
        )
        audio_config = texttospeech.AudioConfig(
            audio_encoding=texttospeech.AudioEncoding.MP3,
            speaking_rate=1.05,  # slightly faster for conversational feel
        )
        
        response = self.tts_client.synthesize_speech(
            input=synthesis_input, voice=voice, audio_config=audio_config
        )
        
        # Save and play
        with open("response.mp3", "wb") as f:
            f.write(response.audio_content)
        
        # Play audio (works on most systems)
        try:
            subprocess.run(["ffplay", "-nodisp", "-autoexit", "response.mp3"],
                         capture_output=True)
        except:
            print("    (Audio saved as response.mp3 — play manually)")
    
    def conversation_loop(self, rounds=3):
        """Run a multi-turn voice conversation."""
        print("Voice Assistant ready! Speak after each prompt.\n")
        
        for i in range(rounds):
            print(f"─── Turn {i+1}/{rounds} ───")
            
            # Listen
            audio = self.listen(duration=5)
            
            # Think
            response_text = self.think(audio)
            print(f"  AI: {response_text}")
            
            # Speak
            self.speak(response_text)
            print()
        
        print("Conversation complete!")

# ── Run it! ──
assistant = VoiceAssistant(
    persona="a friendly AI tutor for the Netsetos GenAI course. You explain AI concepts using simple Indian analogies"
)

assistant.conversation_loop(rounds=3)


> **What just happened?** We built a complete voice AI assistant with a 3-method pipeline: listen() records from the microphone, think() sends audio directly to Gemini (which understands speech natively), speak() converts the response to natural speech with Google TTS. The conversation_loop() runs multi-turn voice conversations. This is the same architecture behind Alexa, Siri, and Google Assistant — but built in ~60 lines of Python. The assistant maintains conversation context across turns because it uses Gemini's start_chat() .


---

## Wrap-up

You've walked through all 7 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
